<a id="tsv-loader"></a>
# Chargement du corpus Bayelemabaga

En conditions réelles, les données ne sont pas écrites en dur dans le code : elles viennent d'un fichier au format CoNLL/TSV, une paire mot \t tag par ligne, une ligne vide séparant les phrases. C'est le format de bambara_pos_prep.conll.

In [1]:
from pathlib import Path

def load_bambara_corpus(file_path):
    """
    Lit un fichier CoNLL/TSV (mot <TAB> tag, ligne vide = fin de phrase)
    et retourne une liste de tuples (mots, tags).
    """
    sentences = []
    current_words, current_tags = [], []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                if current_words:
                    assert len(current_words) == len(current_tags), \
                        f"Désalignement mots/tags à la ligne {line_num}"
                    sentences.append((current_words, current_tags))
                    current_words, current_tags = [], []
                continue

            parts = line.split("\t")
            if len(parts) == 2:
                current_words.append(parts[0])
                current_tags.append(parts[1])
            else:
                print(f"Ligne {line_num} ignorée (format inattendu) : {line!r}")

        if current_words:
            sentences.append((current_words, current_tags))

    print(f"Corpus chargé : {len(sentences)} phrases.")
    return sentences


candidates = [
    Path("bambara_pos_prep_retagged.conll"),
   # Path("bambara-pos-tagging/bambara_pos_prep.conll"),
   # Path.cwd().parent / "bambara_pos_prep.conll",
]

corpus_path = next((path for path in candidates if path.is_file()), None)

if corpus_path is None:
    raise FileNotFoundError(
        "Fichier bambara_pos_prep_retagged.conll introuvable depuis : "
        f"{Path.cwd()}"
    )

corpus_data = load_bambara_corpus(corpus_path)

Corpus chargé : 37392 phrases.


In [2]:
# Statistiques simples sur le corpus
from collections import Counter

nb_sentences = len(corpus_data)
nb_tokens = sum(len(words) for words, _ in corpus_data)
tag_counter = Counter(tag for _, tags in corpus_data for tag in tags)

print(f"Nombre de phrases : {nb_sentences}")
print(f"Nombre de tokens  : {nb_tokens}")
print(f"Nombre de tags distincts : {len(tag_counter)}")
print("\nFréquence des tags :")
for tag, count in tag_counter.most_common():
    print(f"  {tag:<8} : {count}")

print("\nExemple de phrase (index 0) :")
print("Mots :", corpus_data[0][0])
print("Tags :", corpus_data[0][1])

Nombre de phrases : 37392
Nombre de tokens  : 639614
Nombre de tags distincts : 11

Fréquence des tags :
  NOM      : 184133
  PUNCT    : 104611
  AUX      : 97075
  PRON     : 91797
  VERBE    : 63984
  POSTP    : 33411
  CONJ     : 27729
  DET      : 24362
  PART     : 7486
  ADV      : 2599
  ADJ      : 2427

Exemple de phrase (index 0) :
Mots : ['Mieru', 'Baa', 'ka', 'maana', '.', 'Ayiwa', '!']
Tags : ['NOM', 'NOM', 'AUX', 'VERBE', 'PUNCT', 'NOM', 'PUNCT']
